# Google Search Ranking & Discoverability Capstone

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labanaprince72-a11y/internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

## Refresh / Content Opportunity Scoring

**Research question:** Which observable content signals can support a transparent, human-reviewed review queue for content refresh decisions, without treating a retrospective outcome as a production feature?

This capstone extends ML-07 through ML-10. It compares a transparent baseline with a grouped client-holdout model, audits leakage, and turns the validated evidence into ranked decision-support recommendations. The output is not an autonomous publishing system and does not claim to predict or control a search engine.

## 1. Question and decision

The supported decision is: **which anonymized content items should an editor or SEO reviewer inspect first, and what kind of review should happen next?**

Unit of analysis: one anonymized content item. Output: a rank, priority tier, reason code, next step, effort estimate, and human-review gate. A wrong call can waste editorial time or cause an unnecessary content change, so the queue is intentionally conservative and reversible.

In [1]:
from pathlib import Path
import json
import platform
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
OUT_DIR = ROOT / "work/outputs"
FIG_DIR = ROOT / "work/figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists(), f"Expected {DATA_PATH}; run from the repository root."

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} anonymized content rows and {df.shape[1]} columns.")
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

Loaded 30,000 anonymized content rows and 44 columns.
Python 3.12.7 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


## 2. Data and safety

This analysis uses the anonymized starter release shipped with the repository: one row per pseudonymized content item with trailing-90-day search/engagement measures and static content metadata. The starter slice is a snapshot, not a longitudinal warehouse study.

The analysis excludes client names, domains, URLs, search queries, and credentials. `client_id` is used only for the grouped split. The retrospective fields `trend_direction` and `trend_pct` are used only to create an audit target and are never features for the model or the action queue.

In [2]:
required = {
    "content_id", "client_id", "content_type", "trend_direction", "trend_pct",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update", "search_volume", "competition",
    "word_count", "char_count", "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "cpc", "main_intent", "competition_level"
}
missing = sorted(required - set(df.columns))
assert not missing, f"Missing required columns: {missing}"

TARGET = "observed_decline_outcome"
df[TARGET] = (df["trend_direction"] == "down").astype(int)
FORBIDDEN = {"trend_direction", "trend_pct", TARGET, "is_declining_label"}

FEATURES = [
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update", "search_volume",
    "competition", "word_count", "char_count",
]
assert not FORBIDDEN.intersection(FEATURES)
assert "client_id" not in FEATURES and "content_id" not in FEATURES

profile = pd.DataFrame({
    "measure": ["rows", "clients", "decline audit base rate", "median impressions 90d", "median clicks 90d"],
    "value": [len(df), df["client_id"].nunique(), f"{df[TARGET].mean():.1%}", df["impressions_90d"].median(), df["clicks_90d"].median()]
})
print(profile.to_string(index=False))
print(f"\nFinal features ({len(FEATURES)}): {FEATURES}")
print(f"Forbidden fields excluded from the model: {sorted(FORBIDDEN)}")

# The signal table is descriptive only; it does not claim that changing a signal causes an outcome.
df["visibility_bucket"] = pd.cut(
    df["impressions_90d"], bins=[-1, 299, 2999, np.inf],
    labels=["low (<300)", "moderate (300–2,999)", "high (3,000+)"]
).astype(str)
signal_summary = (
    df.groupby("visibility_bucket", observed=False)
      .agg(rows=("content_id", "size"), median_impressions=("impressions_90d", "median"), median_clicks=("clicks_90d", "median"), median_ctr_pct=("ctr", "median"))
      .reset_index()
)
print("\nDescriptive visibility signal summary:")
print(signal_summary.to_string(index=False))

                measure  value
                   rows  30000
                clients     32
decline audit base rate  54.2%
 median impressions 90d  731.0
      median clicks 90d    1.0

Final features (9): ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'search_volume', 'competition', 'word_count', 'char_count']
Forbidden fields excluded from the model: ['is_declining_label', 'observed_decline_outcome', 'trend_direction', 'trend_pct']

Descriptive visibility signal summary:
   visibility_bucket  rows  median_impressions  median_clicks  median_ctr_pct
       high (3,000+)  8283              8426.0           20.0            0.21
          low (<300) 11248                31.0            0.0            0.00
moderate (300–2,999) 10469               998.0            1.0            0.12


## 3. Methodology and leakage checks

The target is an audit-only retrospective label: `trend_direction == "down"`. The transparent baseline uses only pre-decision visibility, position/CTR opportunity, and freshness rules. The model uses nine prior-window or static fields listed above.

The primary validation is a 75/25 `GroupShuffleSplit` by `client_id` with `random_state=42`, so held-out clients do not appear in training. Precision@50 is the operational ranking metric because the queue is intended to prioritize a small review set. ROC-AUC is included as a broader discrimination measure. A random split is shown only as a comparison because repeated-client patterns can make it optimistic.

In [3]:
def precision_at_k(y_true, scores, k):
    y_arr = np.asarray(y_true)
    score_arr = np.asarray(scores)
    order = np.argsort(-score_arr, kind="mergesort")[:k]
    return float(y_arr[order].mean())

def make_logistic():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)),
    ])

def make_forest():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", RandomForestClassifier(n_estimators=160, max_depth=10, min_samples_leaf=10, class_weight="balanced_subsample", random_state=42, n_jobs=-1)),
    ])

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df["client_id"].copy()

# Reconstruct the transparent Week-4 baseline on the same rows.
visible = df["impressions_90d"] >= 300
high_visibility = df["impressions_90d"] >= 3000
position_1_to_20 = df["avg_position"].between(1, 20)
low_ctr = df["ctr"] <= 1.0
stale = df["days_since_last_update"] >= 180
baseline_score = (
    np.select([high_visibility, visible], [2, 1], default=0)
    + np.select([visible & position_1_to_20 & low_ctr, visible & position_1_to_20], [2, 1], default=0)
    + np.select([stale, df["days_since_last_update"] >= 90], [2, 1], default=0)
)

random_train, random_test = train_test_split(np.arange(len(df)), test_size=0.25, random_state=42, stratify=y)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
group_train, group_test = next(gss.split(X, y, groups=groups))
train_clients = set(groups.iloc[group_train])
test_clients = set(groups.iloc[group_test])
assert train_clients.isdisjoint(test_clients)

models = {"logistic_regression": make_logistic(), "random_forest": make_forest()}
results = []
model_predictions = {}
for split_name, train_idx, test_idx in [("random", random_train, random_test), ("grouped_client_holdout", group_train, group_test)]:
    y_test = y.iloc[test_idx]
    base_scores = baseline_score[test_idx]
    results.append({
        "split": split_name, "method": "week4_transparent_baseline",
        "precision_at_10": precision_at_k(y_test, base_scores, 10),
        "precision_at_50": precision_at_k(y_test, base_scores, 50),
        "precision_at_100": precision_at_k(y_test, base_scores, 100),
        "roc_auc": float(roc_auc_score(y_test, base_scores)),
        "accuracy_at_0.5": None,
    })
    for name, model in models.items():
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        prob = model.predict_proba(X.iloc[test_idx])[:, 1]
        model_predictions[(split_name, name)] = prob
        results.append({
            "split": split_name, "method": name,
            "precision_at_10": precision_at_k(y_test, prob, 10),
            "precision_at_50": precision_at_k(y_test, prob, 50),
            "precision_at_100": precision_at_k(y_test, prob, 100),
            "roc_auc": float(roc_auc_score(y_test, prob)),
            "accuracy_at_0.5": float(accuracy_score(y_test, prob >= 0.5)),
        })

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
selected_model_name = "logistic_regression"
selected_prob = model_predictions[("grouped_client_holdout", selected_model_name)]
grouped_y = y.iloc[group_test]
selected_model = models[selected_model_name]
selected_pred = (selected_prob >= 0.5).astype(int)
print(f"\nSelected model: {selected_model_name}")
print(f"Grouped train/test rows: {len(group_train):,}/{len(group_test):,}; clients: {len(train_clients)}/{len(test_clients)}; overlap: {len(train_clients & test_clients)}")
print(f"Grouped threshold precision={precision_score(grouped_y, selected_pred):.3f}; recall={recall_score(grouped_y, selected_pred):.3f}; errors={(selected_pred != grouped_y.to_numpy()).sum():,}")

perm = permutation_importance(selected_model, X.iloc[group_test], grouped_y, scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=-1)
importance = pd.DataFrame({"feature": FEATURES, "mean_importance": perm.importances_mean, "std_importance": perm.importances_std}).sort_values("mean_importance", ascending=False).reset_index(drop=True)
print("\nHeld-out permutation-importance ranking:")
print(importance.head(6).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Intentional target-copy canary: a leakage detector, never a legal feature.
canary = pd.DataFrame({"legal_feature": X["content_age_days"], "intentional_target_copy": y})
canary_model = make_logistic()
canary_model.fit(canary.iloc[group_train], y.iloc[group_train])
canary_prob = canary_model.predict_proba(canary.iloc[group_test])[:, 1]
canary_auc = float(roc_auc_score(y.iloc[group_test], canary_prob))
assert canary_auc > 0.99
print(f"\nIntentional target-copy canary ROC-AUC: {canary_auc:.3f}; canary excluded from final model.")

                 split                     method  precision_at_10  precision_at_50  precision_at_100  roc_auc  accuracy_at_0.5
                random week4_transparent_baseline            0.700            0.520             0.540    0.579              NaN
                random        logistic_regression            0.500            0.680             0.730    0.642            0.623
                random              random_forest            0.900            0.900             0.920    0.774            0.702
grouped_client_holdout week4_transparent_baseline            0.600            0.440             0.430    0.500              NaN
grouped_client_holdout        logistic_regression            0.400            0.620             0.610    0.535            0.532
grouped_client_holdout              random_forest            0.600            0.420             0.490    0.652            0.611

Selected model: logistic_regression
Grouped train/test rows: 22,885/7,115; clients: 24/8; overlap: 0
Gr


Held-out permutation-importance ranking:
             feature  mean_importance  std_importance
    content_age_days           0.0368          0.0054
     clicks_prev_30d           0.0025          0.0011
          word_count           0.0011          0.0017
   sessions_prev_30d           0.0006          0.0006
       search_volume           0.0005          0.0018
impressions_prev_30d           0.0002          0.0008

Intentional target-copy canary ROC-AUC: 1.000; canary excluded from final model.


## 4. Results versus baseline

The grouped client-holdout result is the primary estimate for new-client portability. The model is used as a ranking aid, not a truth oracle. In this snapshot, the selected logistic model ranks the top 50 held-out rows with higher retrospective precision than the transparent baseline, while its ROC-AUC remains modest. That combination supports cautious prioritization, not a claim that edits will cause improvement.

In [4]:
grouped_table = comparison[comparison["split"] == "grouped_client_holdout"].copy()
print(grouped_table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = grouped_table.set_index("method")[["precision_at_50"]]
plot_df.plot(kind="bar", ax=ax, color=["#6b7280", "#4f46e5", "#0f766e"])
ax.set_title("Grouped client-holdout Precision@50")
ax.set_ylabel("Retrospective precision")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.legend(["Precision@50"], loc="upper right")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "capstone_model_vs_baseline.png", dpi=160)
plt.close(fig)
print(f"Saved {FIG_DIR / 'capstone_model_vs_baseline.png'}")

                 split                     method  precision_at_10  precision_at_50  precision_at_100  roc_auc  accuracy_at_0.5
grouped_client_holdout week4_transparent_baseline            0.600            0.440             0.430    0.500              NaN
grouped_client_holdout        logistic_regression            0.400            0.620             0.610    0.535            0.532
grouped_client_holdout              random_forest            0.600            0.420             0.490    0.652            0.611
Saved /Users/macbook/Library/Application Support/strawberry/Synced/c40f8001-9394-43cb-8129-ba077322b099/default/sessions/41c9bec4-ef3e-4062-a50b-77b850d645ab/internship-repo/work/figures/capstone_model_vs_baseline.png


## 5. Limitations and honest framing

- This is a 30,000-row anonymized snapshot with a retrospective decline label; it is not an intervention study.
- The grouped split tests portability across pseudonymous clients in this slice, not future performance across all accounts.
- A model score does not establish page quality, intent fit, or a causal refresh opportunity.
- The `clicks × CPC` field used in the action queue is a captured click-equivalent value proxy, not revenue.
- Changes may be rejected for editorial, legal, accessibility, brand, technical, or client reasons.
- A fresh time-ordered follow-up window is required before any production decision. The playbook therefore requires human review for every row.

In [5]:
# Explicit self-checks for honest use.
assert len(df) == 30000
assert len(FEATURES) == 9
assert not FORBIDDEN.intersection(FEATURES)
assert train_clients.isdisjoint(test_clients)
assert canary_auc > 0.99
assert comparison[comparison["split"] == "grouped_client_holdout"]["precision_at_50"].notna().all()
print("Limitation and safety checks: PASS")
print("No causal claim is made; all recommendations remain decision-support and require human review.")

Limitation and safety checks: PASS
No causal claim is made; all recommendations remain decision-support and require human review.


## 6. Ranked recommendations and action playbook

The action queue uses the transparent Week-4 score as its operational source and the ML-08/ML-09 results as guardrails. It is intentionally interpretable: each row has a reason code, priority tier, recommended next step, estimated effort, value band, and human-review gate. No row authorizes autonomous publishing, deletion, redirects, indexing changes, or other irreversible action.

In [6]:
queue_path = OUT_DIR / "ml10_action_queue.csv"
queue = pd.read_csv(queue_path)
required_queue = {"playbook_rank", "action", "reason_code", "priority_tier", "recommended_next_step", "estimated_effort_hours", "value_band", "human_review_gate", "no_go_without_human"}
assert required_queue.issubset(queue.columns)
assert len(queue) == 30000
assert queue["no_go_without_human"].all()

priority_summary = queue.groupby("priority_tier", observed=False).agg(rows=("playbook_rank", "size"), median_effort_hours=("estimated_effort_hours", "median"), total_value_proxy=("captured_value_proxy_90d", "sum")).reset_index()
print(priority_summary.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print("\nTop action counts:")
print(queue["action"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(8, 4.5))
order = ["P1_review_this_week", "P2_review_next_cycle", "P3_monitor_or_sample", "P4_hold_without_new_evidence"]
counts = queue["priority_tier"].value_counts().reindex(order)
colors = ["#dc2626", "#d97706", "#2563eb", "#6b7280"]
counts.plot(kind="bar", ax=ax, color=colors)
ax.set_title("Human-reviewed action queue by priority")
ax.set_ylabel("Rows")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "capstone_priority_tiers.png", dpi=160)
plt.close(fig)
print(f"Saved {FIG_DIR / 'capstone_priority_tiers.png'}")

# Descriptive signal figure for the paper.
fig, ax = plt.subplots(figsize=(8, 4.5))
signal_summary.plot(x="visibility_bucket", y="median_clicks", kind="bar", ax=ax, color="#0f766e", legend=False)
ax.set_title("Median 90-day clicks by visibility bucket")
ax.set_ylabel("Median clicks")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "capstone_visibility_signal.png", dpi=160)
plt.close(fig)
print(f"Saved {FIG_DIR / 'capstone_visibility_signal.png'}")

               priority_tier  rows  median_effort_hours  total_value_proxy
         P1_review_this_week  2182                 1.50           35072.04
        P2_review_next_cycle 12008                 1.50           84061.47
        P3_monitor_or_sample  6673                 0.50            6198.16
P4_hold_without_new_evidence  9137                 0.25             888.12

Top action counts:
action
improve_snippet_or_intent    12597
monitor                      11248
monitor_or_research           5556
review_search_fit              577
refresh_and_recheck             22


Saved /Users/macbook/Library/Application Support/strawberry/Synced/c40f8001-9394-43cb-8129-ba077322b099/default/sessions/41c9bec4-ef3e-4062-a50b-77b850d645ab/internship-repo/work/figures/capstone_priority_tiers.png
Saved /Users/macbook/Library/Application Support/strawberry/Synced/c40f8001-9394-43cb-8129-ba077322b099/default/sessions/41c9bec4-ef3e-4062-a50b-77b850d645ab/internship-repo/work/figures/capstone_visibility_signal.png


## 7. Artifacts the paper embeds

The paper uses the grouped-holdout comparison figure, the queue-priority figure, and the descriptive visibility figure. The large action CSV is regenerated by the ML-10 notebook and remains intentionally ignored by git; committed JSON receipts and figures provide the traceable summary.

In [7]:
def json_safe(value):
    if value is None:
        return None
    if isinstance(value, (float, np.floating)) and not np.isfinite(value):
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value

comparison_records = [
    {key: json_safe(value) for key, value in row.items()}
    for row in comparison.to_dict(orient="records")
]

metrics = {
    "rows": int(len(df)),
    "lane": "Refresh / Content Opportunity Scoring",
    "target": TARGET,
    "base_rate": float(y.mean()),
    "features": FEATURES,
    "forbidden_inputs_excluded": sorted(FORBIDDEN),
    "random_split": [row for row in comparison_records if row["split"] == "random"],
    "grouped_client_holdout": [row for row in comparison_records if row["split"] == "grouped_client_holdout"],
    "grouped_train_clients": int(len(train_clients)),
    "grouped_test_clients": int(len(test_clients)),
    "client_overlap": int(len(train_clients & test_clients)),
    "selected_model": selected_model_name,
    "intentional_target_copy_canary_roc_auc": canary_auc,
    "queue_rows": int(len(queue)),
    "queue_priority_counts": {str(k): int(v) for k, v in queue["priority_tier"].value_counts().sort_index().items()},
    "queue_action_counts": {str(k): int(v) for k, v in queue["action"].value_counts().items()},
    "p1_click_value_proxy_total": float(queue.loc[queue["priority_tier"] == "P1_review_this_week", "captured_value_proxy_90d"].sum()),
    "all_rows_require_human_review": bool(queue["no_go_without_human"].all()),
    "figures": [
        "work/figures/capstone_model_vs_baseline.png",
        "work/figures/capstone_priority_tiers.png",
        "work/figures/capstone_visibility_signal.png",
    ],
}
metrics_path = OUT_DIR / "ml_capstone_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, allow_nan=False))
for figure in metrics["figures"]:
    assert (ROOT / figure).exists()
print(f"Wrote {metrics_path}")
print("Artifact self-check: PASS")

Wrote /Users/macbook/Library/Application Support/strawberry/Synced/c40f8001-9394-43cb-8129-ba077322b099/default/sessions/41c9bec4-ef3e-4062-a50b-77b850d645ab/internship-repo/work/outputs/ml_capstone_metrics.json
Artifact self-check: PASS


## 8. Reproducibility

From a fresh clone, install the repository requirements and execute this notebook from the repository root:

```bash
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute work/notebooks/capstone.ipynb --output work/notebooks/capstone.ipynb --ExecutePreprocessor.timeout=600
```

The split seed is `42`. The primary split is grouped by `client_id`; the identifier is never a model feature. The notebook writes `work/outputs/ml_capstone_metrics.json` and three reusable figures. The ML-10 notebook regenerates the ignored `work/outputs/ml10_action_queue.csv` before the capstone is run.

In [8]:
# Final reproducibility checklist.
for path in [
    OUT_DIR / "ml07_baseline_metrics.json",
    OUT_DIR / "ml08_model_metrics.json",
    OUT_DIR / "ml09_validation_metrics.json",
    OUT_DIR / "ml10_playbook_metrics.json",
    OUT_DIR / "ml_capstone_metrics.json",
]:
    assert path.exists(), path
print("Reproducibility receipt check: PASS")
print("The capstone is complete, executed, and ready to be paired with the deployed paper.")

Reproducibility receipt check: PASS
The capstone is complete, executed, and ready to be paired with the deployed paper.


## 9. ML-12 closing cells

### Five-minute demo outline

1. State the decision: prioritize human review of anonymized content opportunities.
2. Show the transparent baseline and why retrospective fields are excluded from features.
3. Show the grouped client-holdout comparison: baseline versus model at Precision@50.
4. Open the action queue: priority, reason code, next step, effort, value proxy, and review gate.
5. Close with limits: this is decision-support, not autonomous publishing or causal proof.

### Social-post cut

I built an honest, human-reviewed content opportunity scoring workflow from anonymized FlyRank data. The project compares a transparent baseline with a grouped client-holdout model, checks leakage, and turns the result into a practical review queue. The main lesson: a useful score is only the beginning — the guardrails and review process matter just as much.

### Employer-facing summary

I built and documented a reproducible applied-ML workflow that moves from problem framing and data safety through grouped validation and a ranked action playbook. I can explain both the measured signal and its limits, including why a higher retrospective ranking metric does not prove that a content change will cause improvement. The resulting paper and repository show practical Python, pandas, scikit-learn, evaluation, visualization, and communication skills.

## Acknowledgments and data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai). The public paper contains aggregate findings and decision-support guidance only; it does not expose private client identifiers, domains, URLs, queries, credentials, or raw exports.